In [17]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [18]:
import pyarrow.parquet as pq
trips = pq.read_table('Trips_phase3.parquet')
trips = trips.to_pandas()

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn import set_config
from sklearn.preprocessing import FunctionTransformer


In [20]:
target = 'fare_amount'
y = trips[target]
x = trips.drop(columns = [target])

X_train,X_test , y_train,y_test = train_test_split(x ,y,test_size = 0.2 ,random_state = 42)

In [21]:
# Ensure scikit-learn transformers output DataFrames, preserving dtypes
set_config(transform_output="pandas")

def cast_taxi_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # 1. Float conversions
    float_cols = ['trip_distance', 'duration']
    for col in float_cols:
        if col in df.columns:
            df[col] = df[col].astype('float32')
            
    # 2. Integer conversions (Int8 handles nullable integers)
    if 'RatecodeID' in df.columns:
        df['RatecodeID'] = df['RatecodeID'].astype('Int8')
        
    int8_cols = ['day_of_the_week', 'pick_up_month']
    for col in int8_cols:
        if col in df.columns:
            df[col] = df[col].astype('int8')
            
    # 3. Boolean conversions
    bool_cols = [
        'is_weekend', 'is_rush_hour', 'is_late_night',
        'is_airport_pickup', 'is_airport_dropoff', 'SameZone'
    ]
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].astype('bool')
            
    # 4. Categorical conversions (XGBoost can consume these natively)
    cat_cols = ['boroughPickUpPoint', 'boroughDropOffPoint']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    return df

In [22]:
type_converter = FunctionTransformer(cast_taxi_dtypes)
numeric_cols = ['trip_distance', 'duration']
categorical_cols = ['RatecodeID', 'boroughPickUpPoint', 'boroughDropOffPoint']

# 2. Single preprocessor handling both scaling and one-hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('scale', StandardScaler(), numeric_cols),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough'  # Keeps boolean flags and remaining integer columns untouched
)

In [23]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1899292 entries, 347833 to 2474701
Data columns (total 13 columns):
 #   Column               Dtype   
---  ------               -----   
 0   trip_distance        float32 
 1   RatecodeID           Int8    
 2   duration             float32 
 3   day_of_the_week      int8    
 4   pick_up_month        int8    
 5   is_weekend           bool    
 6   is_rush_hour         bool    
 7   is_late_night        bool    
 8   boroughPickUpPoint   category
 9   boroughDropOffPoint  category
 10  is_airport_pickup    bool    
 11  is_airport_dropoff   bool    
 12  SameZone             bool    
dtypes: Int8(1), bool(6), category(2), float32(2), int8(2)
memory usage: 50.7 MB


<h3>Model #1 Baseline Linear regressor</h3>

In [8]:
model = Pipeline(steps=[
    ('type_converter', type_converter),
    ('features', preprocessor),
    ('regressor', LinearRegression())
])


model.fit(X_train, y_train)


y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

RMSE: 4.536
R²:   0.939


In [12]:
!pip install xgboost


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Python311\python.exe -m pip install --upgrade pip


In [15]:
import xgboost as xgb

<h3>Model #2 XGBoost</h3>

In [11]:

full_XGB_pipeline = Pipeline(steps=[
    ('type_converter', type_converter),
    ('features', preprocessor),
    ('XGBRegressor', xgb.XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        n_jobs=-1,
        random_state=42
    ))
])

full_XGB_pipeline.fit(X_train, y_train)
y_pred_xgb = full_XGB_pipeline.predict(X_test)


In [12]:
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)
print(f"XGB RMSE: {rmse_xgb:.3f} | R²: {r2_xgb:.3f}")

XGB RMSE: 3.482 | R²: 0.964


<h3>Model #3 RandomForest</h3>

In [26]:
from sklearn.ensemble import RandomForestRegressor

In [14]:
full_RF_pipeline = Pipeline(steps=[
    ('type_converter', type_converter),
    ('features', preprocessor),
    ('RFRegressor', RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
    ))
])

# Fit and evaluate
full_RF_pipeline.fit(X_train, y_train)
y_pred_RF = full_RF_pipeline.predict(X_test)


In [ ]:
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_RF))
r2_xgb = r2_score(y_test, y_pred_RF)
print(f"Random Forest RMSE: {rmse_xgb:.3f} | R²: {r2_xgb:.3f}")

XGB RMSE: 2.754 | R²: 0.977


<h4>Saving the Models</h4>

In [30]:
import joblib

In [ ]:


# Save each model
joblib.dump(model, 'Models/linear_regression_model.pkl')      # your Pipeline from earlier
joblib.dump(full_RF_pipeline, 'Models/random_forest_model.pkl')
joblib.dump(full_XGB_pipeline, 'Models/xgboost_model.pkl')             # or hgb if using HistGradientBoostingRegressor

['Models/xgboost_model.pkl']

<h3>Hyperparameter Fine Tuning</h3>

In [10]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

def evaluate_pipeline(pipeline, X_test, y_test, name="Model"):
    preds = pipeline.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"{name} -> Best Test RMSE: {rmse:.3f} | R²: {r2:.3f}")
    return rmse, r2

In [24]:
xgb_base = Pipeline(steps=[
    ('type_converter', type_converter),
    ('features', preprocessor),
    ('regressor', xgb.XGBRegressor(
        tree_method='hist',
        n_jobs=-1,
        random_state=42
    ))
])


xgb_param_dist = {
    'regressor__n_estimators': randint(300, 800),
    'regressor__max_depth': randint(5, 12),
    'regressor__learning_rate': uniform(0.02, 0.15),      
    'regressor__subsample': uniform(0.65, 0.35),          
    'regressor__colsample_bytree': uniform(0.65, 0.35),
    'regressor__reg_alpha': uniform(0.0, 5.0),            
    'regressor__reg_lambda': uniform(1.0, 9.0)          
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=15,                          
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1                           
)

print("Starting XGBoost hyperparameter search...")
xgb_search.fit(X_train, y_train)

best_xgb_pipeline = xgb_search.best_estimator_
print(f"Best XGB Params: {xgb_search.best_params_}")
evaluate_pipeline(best_xgb_pipeline, X_test, y_test, name="Tuned XGBoost")

Starting XGBoost hyperparameter search...
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[CV] END regressor__colsample_bytree=0.7810890415965769, regressor__learning_rate=0.1626071459614874, regressor__max_depth=7, regressor__n_estimators=371, regressor__reg_alpha=2.993292420985183, regressor__reg_lambda=2.4041677639819286, regressor__subsample=0.7045980821176709; total time=  23.5s


e:\CONDA\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END regressor__colsample_bytree=0.7810890415965769, regressor__learning_rate=0.1626071459614874, regressor__max_depth=7, regressor__n_estimators=371, regressor__reg_alpha=2.993292420985183, regressor__reg_lambda=2.4041677639819286, regressor__subsample=0.7045980821176709; total time=  20.8s
[CV] END regressor__colsample_bytree=0.7810890415965769, regressor__learning_rate=0.1626071459614874, regressor__max_depth=7, regressor__n_estimators=371, regressor__reg_alpha=2.993292420985183, regressor__reg_lambda=2.4041677639819286, regressor__subsample=0.7045980821176709; total time=  23.0s
[CV] END regressor__colsample_bytree=0.6703292642588699, regressor__learning_rate=0.14992642186624025, regressor__max_depth=8, regressor__n_estimators=659, regressor__reg_alpha=3.540362888980227, regressor__reg_lambda=1.185260448662222, regressor__subsample=0.989468448256698; total time=  37.4s


e:\CONDA\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END regressor__colsample_bytree=0.6703292642588699, regressor__learning_rate=0.14992642186624025, regressor__max_depth=8, regressor__n_estimators=659, regressor__reg_alpha=3.540362888980227, regressor__reg_lambda=1.185260448662222, regressor__subsample=0.989468448256698; total time=  35.8s
[CV] END regressor__colsample_bytree=0.6703292642588699, regressor__learning_rate=0.14992642186624025, regressor__max_depth=8, regressor__n_estimators=659, regressor__reg_alpha=3.540362888980227, regressor__reg_lambda=1.185260448662222, regressor__subsample=0.989468448256698; total time=  44.7s
[CV] END regressor__colsample_bytree=0.9413549242801476, regressor__learning_rate=0.051850866601741416, regressor__max_depth=8, regressor__n_estimators=576, regressor__reg_alpha=3.0874075481385828, regressor__reg_lambda=6.504878444394528, regressor__subsample=0.6524732068269011; total time=  38.9s


...

e:\CONDA\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END regressor__colsample_bytree=0.6905541708337954, regressor__learning_rate=0.14946551388133902, regressor__max_depth=8, regressor__n_estimators=717, regressor__reg_alpha=0.47705058245205656, regressor__reg_lambda=4.3373642697843975, regressor__subsample=0.8840944384322625; total time=  44.1s
[CV] END regressor__colsample_bytree=0.6905541708337954, regressor__learning_rate=0.14946551388133902, regressor__max_depth=8, regressor__n_estimators=717, regressor__reg_alpha=0.47705058245205656, regressor__reg_lambda=4.3373642697843975, regressor__subsample=0.8840944384322625; total time=  44.3s
[CV] END regressor__colsample_bytree=0.8830728248161238, regressor__learning_rate=0.10869466815615907, regressor__max_depth=10, regressor__n_estimators=774, regressor__reg_alpha=2.806217129238506, regressor__reg_lambda=4.446341872784108, regressor__subsample=0.9900992333861863; total time=  53.2s


e:\CONDA\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END regressor__colsample_bytree=0.8830728248161238, regressor__learning_rate=0.10869466815615907, regressor__max_depth=10, regressor__n_estimators=774, regressor__reg_alpha=2.806217129238506, regressor__reg_lambda=4.446341872784108, regressor__subsample=0.9900992333861863; total time=  50.9s
[CV] END regressor__colsample_bytree=0.8830728248161238, regressor__learning_rate=0.10869466815615907, regressor__max_depth=10, regressor__n_estimators=774, regressor__reg_alpha=2.806217129238506, regressor__reg_lambda=4.446341872784108, regressor__subsample=0.9900992333861863; total time= 1.0min
Best XGB Params: {'regressor__colsample_bytree': np.float64(0.848695114736997), 'regressor__learning_rate': np.float64(0.024696993868333787), 'regressor__max_depth': 6, 'regressor__n_estimators': 517, 'regressor__reg_alpha': np.float64(2.248770666848828), 'regressor__reg_lambda': np.float64(4.55635212401633), 'regressor__subsample': np.float64(0.974330603027828)}
Tuned XGBoost -> Best Test RMSE: 3.440

(np.float64(3.4399158658229463), 0.964905321598053)

In [28]:
rf_base = Pipeline(steps=[
    ('type_converter', type_converter),
    ('features', preprocessor),
    ('regressor', RandomForestRegressor(
        n_jobs=-1,
        random_state=42
    ))
])

rf_param_dist = {
    'regressor__n_estimators': [150, 250, 350],
    'regressor__max_depth': [12, 16, 20, 25],
    'regressor__min_samples_split': [5, 10, 20],
    'regressor__min_samples_leaf': [2, 5, 10],
    'regressor__max_features': ['sqrt', 0.6, 0.8],
    'regressor__max_samples': [0.4, 0.6, 0.8]             # Caps rows sampled per tree to speed up training
}
# 1. Sample 250k rows for search
X_sample, _, y_sample, _ = train_test_split(
    X_train, y_train, 
    train_size=250_000, 
    random_state=42
)

# 2. Run RandomizedSearchCV on the sample (completes in 5-8 mins)
rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_dist,
    n_iter=8,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1
)

rf_search.fit(X_sample, y_sample)
print(f"Best RF Params: {rf_search.best_params_}")

# 3. Train ONLY the best estimator once on the full 1.9M rows
best_rf_pipeline = rf_search.best_estimator_
best_rf_pipeline.fit(X_train, y_train)

evaluate_pipeline(best_rf_pipeline, X_test, y_test, name="Tuned Random Forest")

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END regressor__max_depth=12, regressor__max_features=0.6, regressor__max_samples=0.4, regressor__min_samples_leaf=10, regressor__min_samples_split=10, regressor__n_estimators=150; total time=   4.7s
[CV] END regressor__max_depth=12, regressor__max_features=0.6, regressor__max_samples=0.4, regressor__min_samples_leaf=10, regressor__min_samples_split=10, regressor__n_estimators=150; total time=   4.1s
[CV] END regressor__max_depth=12, regressor__max_features=0.6, regressor__max_samples=0.4, regressor__min_samples_leaf=10, regressor__min_samples_split=10, regressor__n_estimators=150; total time=   4.1s
[CV] END regressor__max_depth=16, regressor__max_features=0.8, regressor__max_samples=0.6, regressor__min_samples_leaf=2, regressor__min_samples_split=10, regressor__n_estimators=150; total time=   8.4s
[CV] END regressor__max_depth=16, regressor__max_features=0.8, regressor__max_samples=0.6, regressor__min_samples_leaf=2, reg

(np.float64(2.7261913184216633), 0.9779576249872715)

In [ ]:
joblib.dump(best_rf_pipeline, 'Models/Linear-Regression-TLC-NYC-Fare-Price-Prediction/random_forest_model.pkl')
joblib.dump(best_xgb_pipeline, 'Models/Random-Forest-Regressor-For-TLC-NYC-Fare-Price-Prediction/xgboost_model.pkl')

['Models/xgboost_modelV1.2.pkl']